# Joined dataset and US reporting scope (EDA)

Combines EKPO → EKKO → LFA1, applies the agreed scope and saves the in-scope item table
(`data/processed/po_scope.parquet`) used by every analysis question, so Q1–Q4 share one population.

| Setting | Value |
|---|---|
| Company code / purchasing org | US00 / US00 (filter on purch. org) |
| Plants | all US plants on in-scope POs (MI00, DL00, SD00) |
| Document category | F (purchase orders); deleted items excluded |
| Reporting period | 1 Oct 2025 – 30 Sep 2026 (last 12 complete months, by Document Date) |
| Currency / tool | USD / Python (pandas) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sap_data import load, load_raw, PURCH_ORG, COMPANY_CODE, PERIOD_START, PERIOD_END

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
ekko, ekpo, lfa1 = load("ekko"), load("ekpo"), load("lfa1")

# Category / company code exist on both EKKO and EKPO (verified identical in ekpo_eda); keep header copy.
# Both EKKO and LFA1 have "Created On"; rename to keep them apart.
df = (ekpo.drop(columns=["Purch. Doc. Category", "Company Code"])
      .merge(ekko.rename(columns={"Created On": "PO Created On", "Created By": "PO Created By"}),
             on="Purchasing Document", how="left", validate="many_to_one")
      .merge(lfa1.rename(columns={"Created On": "Supplier Created On"}),
             on="Supplier", how="left", validate="many_to_one"))
assert len(df) == len(ekpo)   # joins did not duplicate items
df.shape

`validate="many_to_one"` makes pandas raise an error if a join key isn't unique on the right-hand
side. Together with the row-count assertion, that guarantees no item is silently duplicated (and no
value double-counted).

## 1. Apply the filters step by step

In [ ]:
steps = {"all EKPO items": df}
steps["category F (POs)"] = s = df[df["Purch. Doc. Category"] == "F"]
steps["not deleted"] = s = s[s["Deletion Indicator"].isna()]
steps[f"purch. org {PURCH_ORG}"] = po = s[s["Purch. Organization"] == PURCH_ORG]
steps["in reporting window"] = w = po[(po["Document Date"] >= PERIOD_START)
                                      & (po["Document Date"] < PERIOD_END)].copy()
pd.DataFrame({k: {"items": len(v), "POs": v["Purchasing Document"].nunique(),
                  "net_value_usd": v["Net Order Value"].sum()} for k, v in steps.items()}).T

In [ ]:
# Alternative scope: also accept POs with blank purch. org but company code US00
alt = s[(s["Purch. Organization"] == PURCH_ORG)
        | (s["Purch. Organization"].isna() & (s["Company Code"] == COMPANY_CODE))]
alt_w = alt[(alt["Document Date"] >= PERIOD_START) & (alt["Document Date"] < PERIOD_END)]
print(f"Company-code fallback adds {len(alt_w) - len(w)} items / "
      f"{alt_w['Net Order Value'].sum() - w['Net Order Value'].sum():,.2f} USD in the window")

> **Note.** The purchasing-org filter vs. a company-code fallback differs by a handful of items and
> negligible value, so the scope choice doesn't change any result.

## 2. Currency and supplier completeness in scope

In [ ]:
display(w["Currency"].value_counts(dropna=False))
display(w.loc[w["Currency"].isna(), ["Purchasing Document", "Company Code", "Net Order Value"]])
print("Items without supplier on header:", w["Supplier"].isna().sum())
print("Items whose supplier is missing from LFA1:", (w["Supplier"].notna() & w["Name 1"].isna()).sum())

> **Note.** All in-scope items are USD except a few with a blank header currency. They belong to company
> code US00 (local currency USD) and are treated as USD. Items with no supplier stay in the totals as
> "(unknown supplier)" so that supplier shares add up to 100% of the scope.

In [ ]:
w["Currency"] = w["Currency"].fillna("USD")
w["Name 1"] = w["Name 1"].fillna("(unknown supplier)")
w["month"] = w["Document Date"].dt.to_period("M")

## 3. What the reporting window excludes

In [ ]:
outside = po[~po.index.isin(w.index)]
display(outside.groupby(outside["Document Date"].dt.to_period("M"))["Net Order Value"]
        .agg(items="size", net_value="sum").tail(6))
top = po.nlargest(1, "Net Order Value")
print(f'Largest PO item in extract: {top["Name 1"].iat[0]}, {top["Document Date"].iat[0].date()}, '
      f'{top["Net Order Value"].iat[0]:,.0f} USD = {top["Net Order Value"].iat[0] / po["Net Order Value"].sum():.0%} '
      f'of all US00 PO value')

> **Note.** The single largest item (Mid-West Supply, chain locks, **$6.4M**, Oct 2026) is in the current
> incomplete month and is excluded by the window. With "all data" it alone would be ~47% of US PO value.
> Mention this in the limitations.

## 4. Distribution of Net PO value in scope

In [ ]:
display(w["Net Order Value"].describe())
print("Zero-value items:", (w["Net Order Value"] == 0).sum())
print(f'Mean {w["Net Order Value"].mean():,.0f} vs median {w["Net Order Value"].median():,.0f} USD')
w.nlargest(10, "Net Order Value")[["Purchasing Document", "Document Date", "Name 1", "Short Text",
                                   "Material Group", "Net Order Value"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(w["Net Order Value"].clip(lower=1), log_scale=True, bins=40, ax=ax)
ax.axvline(w["Net Order Value"].median(), color="C1", ls="--", label="median")
ax.set(title="Net PO value per item, US00, Oct 2025 – Sep 2026 (log scale)",
       xlabel="Net PO value, USD", ylabel="PO items")
ax.legend();

> **Note.** Right-skewed: median ≈ $2.45k, mean ≈ $3.5k, max $150k (flour exercise items). ~27
> zero-value items (free-of-charge / price-unit errors). Report the median next to the mean and use log
> axes.

## 5. Quick profile of the scope (inputs for Q1–Q4)

In [ ]:
print("Items:", len(w), "| POs:", w["Purchasing Document"].nunique(),
      "| Suppliers:", w["Supplier"].nunique(), "| Net PO value:", f'{w["Net Order Value"].sum():,.0f} USD')
for c in ["Country/Region Key", "Material Group", "Plant", "Purchasing Group"]:
    display(w.groupby(c, dropna=False)["Net Order Value"].agg(items="size", net_value="sum")
              .assign(share=lambda d: (d.net_value / d.net_value.sum()).round(3))
              .sort_values("net_value", ascending=False).head(10))

In [ ]:
# Months with no POs must show up as 0, not vanish
months = pd.period_range(PERIOD_START, PERIOD_END - pd.Timedelta(days=1), freq="M")
w.groupby("month").agg(items=("Item", "size"), POs=("Purchasing Document", "nunique"),
                       net_value=("Net Order Value", "sum")).reindex(months, fill_value=0)

In [ ]:
sup_us = w.drop_duplicates("Supplier").query("`Country/Region Key` == 'US'")
w_us = w[w["Country/Region Key"] == "US"]
print(f'US suppliers without state: {sup_us["Region"].isna().mean():.0%} of suppliers, '
      f'{w_us["Region"].isna().mean():.0%} of items, '
      f'{w_us.loc[w_us["Region"].isna(), "Net Order Value"].sum() / w_us["Net Order Value"].sum():.0%} of value')
w["Name 1"].value_counts().head(15)

> **Notes for the report**
> - Several months in the window have **no POs** (Jan–Feb, Jun–Aug 2026). The monthly series must be
>   reindexed so they appear as 0.
> - Material groups in scope: UTIL dominates, then RAW, with BIKES tiny. Q4 needs the material-prefix
>   or short-text mapping from `ekpo_eda`.
> - A large share of US suppliers has no state, so "Unknown state" gets its own bar.
> - Many top "suppliers" are exercise placeholders (`Bike supplier 000`, `Vendor`, `Villám`,
>   `szállító…`). This is a training client, so every finding describes the data, not a real market.

## 6. Save the in-scope table for the analysis notebook

In [ ]:
from sap_data import PROC
w.assign(month=w["month"].astype(str)).to_parquet(PROC / "po_scope.parquet")
print("saved", len(w), "rows ->", PROC / "po_scope.parquet")